# Module 04: Diffusion Models from Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prashantkul/learn-generative-ai/blob/main/04-diffusion-models/notebook.ipynb)

**GPU recommended:** Yes (training DDPM on MNIST takes ~10 min on GPU, ~45 min on CPU).

## 0. Setup and Imports

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": False,
    "figure.dpi": 100,
})

## 1. Load MNIST Data

In [ ]:
IMG_SIZE = 28
BATCH_SIZE = 128

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x * 2) - 1),  # scale to [-1, 1]
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f"Dataset size: {len(train_dataset)}")
print(f"Number of batches: {len(train_loader)}")

In [ ]:
# Grab a sample batch for visualization
sample_batch, sample_labels = next(iter(train_loader))

fig, axes = plt.subplots(1, 8, figsize=(12, 1.5))
for i, ax in enumerate(axes):
    img = (sample_batch[i].squeeze() + 1) / 2  # back to [0, 1]
    ax.imshow(img, cmap="gray")
    ax.set_title(str(sample_labels[i].item()), fontsize=10)
    ax.axis("off")
plt.suptitle("Sample MNIST images", fontsize=12)
plt.tight_layout()
plt.show()

## 2. Noise Schedule

We use a linear beta schedule as in the original DDPM paper (Ho et al., 2020).

Given $\beta_1, \ldots, \beta_T$, we define:
- $\alpha_t = 1 - \beta_t$
- $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$

$\bar{\alpha}_t$ lets us jump directly from $x_0$ to $x_t$ in one step.

In [ ]:
T = 1000  # total diffusion timesteps


def linear_beta_schedule(timesteps, beta_start=1e-4, beta_end=0.02):
    """Linear schedule from beta_start to beta_end."""
    return torch.linspace(beta_start, beta_end, timesteps)


betas = linear_beta_schedule(T)
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)
alpha_bar_prev = F.pad(alpha_bar[:-1], (1, 0), value=1.0)

# Precompute useful quantities
sqrt_alpha_bar = torch.sqrt(alpha_bar)
sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - alpha_bar)
sqrt_recip_alpha = torch.sqrt(1.0 / alphas)
posterior_variance = betas * (1.0 - alpha_bar_prev) / (1.0 - alpha_bar)

print(f"beta range: [{betas[0]:.6f}, {betas[-1]:.6f}]")
print(f"alpha_bar range: [{alpha_bar[-1]:.6f}, {alpha_bar[0]:.6f}]")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

axes[0].plot(betas.numpy(), linewidth=1.5)
axes[0].set_title("Beta schedule")
axes[0].set_xlabel("Timestep t")
axes[0].set_ylabel("beta_t")

axes[1].plot(alpha_bar.numpy(), linewidth=1.5)
axes[1].set_title("Cumulative alpha_bar")
axes[1].set_xlabel("Timestep t")
axes[1].set_ylabel("alpha_bar_t")

axes[2].plot(sqrt_one_minus_alpha_bar.numpy(), linewidth=1.5, label="sqrt(1 - alpha_bar)")
axes[2].plot(sqrt_alpha_bar.numpy(), linewidth=1.5, label="sqrt(alpha_bar)")
axes[2].set_title("Signal vs noise coefficients")
axes[2].set_xlabel("Timestep t")
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Forward Process: $q(x_t | x_0)$

The forward process adds Gaussian noise according to:

$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t)\, \mathbf{I})$$

Using the reparameterization trick:

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, \mathbf{I})$$

In [ ]:
def q_sample(x_0, t, noise=None):
    """Sample x_t from q(x_t | x_0) using the reparameterization trick.

    Args:
        x_0: clean images, shape (B, C, H, W)
        t: timestep indices, shape (B,)
        noise: optional pre-sampled noise

    Returns:
        x_t: noisy images at timestep t
        noise: the noise that was added
    """
    if noise is None:
        noise = torch.randn_like(x_0)

    sqrt_ab = sqrt_alpha_bar[t].view(-1, 1, 1, 1).to(x_0.device)
    sqrt_one_minus_ab = sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1).to(x_0.device)

    x_t = sqrt_ab * x_0 + sqrt_one_minus_ab * noise
    return x_t, noise

### Visualize the Forward Process

Take a single MNIST image and show how it degrades as $t$ increases.

In [ ]:
x_0_single = sample_batch[0:1]  # (1, 1, 28, 28)
timesteps_to_show = [0, 50, 100, 200, 400, 600, 800, 999]

fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(16, 2))
torch.manual_seed(42)

for i, t_val in enumerate(timesteps_to_show):
    t_tensor = torch.tensor([t_val])
    x_t, _ = q_sample(x_0_single, t_tensor)
    img = (x_t.squeeze().clamp(-1, 1) + 1) / 2
    axes[i].imshow(img.numpy(), cmap="gray", vmin=0, vmax=1)
    axes[i].set_title(f"t={t_val}", fontsize=10)
    axes[i].axis("off")

plt.suptitle("Forward diffusion process: progressively adding noise", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Show pixel value distributions at different timesteps
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
torch.manual_seed(42)
selected_t = [0, 100, 500, 999]

for i, t_val in enumerate(selected_t):
    t_tensor = torch.tensor([t_val])
    x_t, _ = q_sample(x_0_single, t_tensor)
    axes[i].hist(x_t.squeeze().numpy().flatten(), bins=50, density=True, alpha=0.7)
    axes[i].set_title(f"t={t_val}")
    axes[i].set_xlim(-3, 3)

plt.suptitle("Pixel value distributions at various timesteps", fontsize=12)
plt.tight_layout()
plt.show()

## 4. U-Net for Noise Prediction

We build a small U-Net that takes a noisy image $x_t$ and timestep $t$, and predicts the noise $\epsilon$.

Key components:
- **Sinusoidal timestep embedding** (same idea as positional encoding in transformers)
- **Encoder** (downsampling path)
- **Decoder** (upsampling path with skip connections)

This is intentionally kept small for MNIST (28x28 grayscale).

In [ ]:
class SinusoidalPositionEmbedding(nn.Module):
    """Encode timestep t into a vector using sinusoidal embeddings."""

    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device, dtype=torch.float32) * -emb)
        emb = t.float().unsqueeze(1) * emb.unsqueeze(0)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)

In [ ]:
class ResBlock(nn.Module):
    """Residual block with timestep conditioning."""

    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.GroupNorm(8, in_ch),
            nn.SiLU(),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
        )
        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, out_ch),
        )
        self.conv2 = nn.Sequential(
            nn.GroupNorm(8, out_ch),
            nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
        )
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(x)
        h = h + self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = self.conv2(h)
        return h + self.skip(x)

In [ ]:
class SimpleUNet(nn.Module):
    """A small U-Net for MNIST noise prediction.

    Architecture:
        Encoder: 1 -> 64 -> 128 (with downsampling)
        Bottleneck: 128 -> 128
        Decoder: 128 -> 64 -> 1 (with upsampling + skip connections)
    """

    def __init__(self, in_channels=1, time_dim=128, base_channels=64):
        super().__init__()
        self.time_dim = time_dim

        # Timestep embedding
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        # Initial convolution
        self.conv_in = nn.Conv2d(in_channels, base_channels, 3, padding=1)

        # Encoder
        self.enc1 = ResBlock(base_channels, base_channels, time_dim)
        self.down1 = nn.Conv2d(base_channels, base_channels, 4, stride=2, padding=1)  # 28->14
        self.enc2 = ResBlock(base_channels, base_channels * 2, time_dim)
        self.down2 = nn.Conv2d(base_channels * 2, base_channels * 2, 4, stride=2, padding=1)  # 14->7

        # Bottleneck
        self.bottleneck = ResBlock(base_channels * 2, base_channels * 2, time_dim)

        # Decoder
        self.up2 = nn.ConvTranspose2d(base_channels * 2, base_channels * 2, 4, stride=2, padding=1)  # 7->14
        self.dec2 = ResBlock(base_channels * 4, base_channels, time_dim)  # concat skip
        self.up1 = nn.ConvTranspose2d(base_channels, base_channels, 4, stride=2, padding=1)  # 14->28
        self.dec1 = ResBlock(base_channels * 2, base_channels, time_dim)  # concat skip

        # Output
        self.conv_out = nn.Sequential(
            nn.GroupNorm(8, base_channels),
            nn.SiLU(),
            nn.Conv2d(base_channels, in_channels, 1),
        )

    def forward(self, x, t):
        t_emb = self.time_embed(t)

        # Encoder
        x = self.conv_in(x)
        h1 = self.enc1(x, t_emb)          # (B, 64, 28, 28)
        x = self.down1(h1)                 # (B, 64, 14, 14)
        h2 = self.enc2(x, t_emb)           # (B, 128, 14, 14)
        x = self.down2(h2)                 # (B, 128, 7, 7)

        # Bottleneck
        x = self.bottleneck(x, t_emb)      # (B, 128, 7, 7)

        # Decoder
        x = self.up2(x)                    # (B, 128, 14, 14)
        x = torch.cat([x, h2], dim=1)      # (B, 256, 14, 14)
        x = self.dec2(x, t_emb)            # (B, 64, 14, 14)
        x = self.up1(x)                    # (B, 64, 28, 28)
        x = torch.cat([x, h1], dim=1)      # (B, 128, 28, 28)
        x = self.dec1(x, t_emb)            # (B, 64, 28, 28)

        return self.conv_out(x)

In [ ]:
model = SimpleUNet(in_channels=1, time_dim=128, base_channels=64).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

# Quick forward pass test
with torch.no_grad():
    dummy_x = torch.randn(2, 1, 28, 28).to(device)
    dummy_t = torch.tensor([0, 500]).to(device)
    out = model(dummy_x, dummy_t)
    print(f"Input shape:  {dummy_x.shape}")
    print(f"Output shape: {out.shape}")

## 5. Training Loop

The DDPM training objective is simple:
1. Sample a clean image $x_0$ from the dataset
2. Sample a random timestep $t \sim \text{Uniform}(0, T-1)$
3. Sample noise $\epsilon \sim \mathcal{N}(0, \mathbf{I})$
4. Compute $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \epsilon$
5. Predict $\hat{\epsilon} = \text{UNet}(x_t, t)$
6. Minimize $\|\epsilon - \hat{\epsilon}\|^2$

In [ ]:
NUM_EPOCHS = 15
LR = 2e-4

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
losses = []
checkpoint_epochs = [1, 5, 10, 15]  # save samples at these epochs
checkpoint_samples = {}  # epoch -> generated samples

In [ ]:
@torch.no_grad()
def ddpm_sample(model, n_samples, img_size=28, channels=1):
    """Generate samples using DDPM reverse process.

    Start from x_T ~ N(0, I) and iteratively denoise for T steps.
    """
    model.eval()
    x = torch.randn(n_samples, channels, img_size, img_size).to(device)

    for t_val in reversed(range(T)):
        t_batch = torch.full((n_samples,), t_val, dtype=torch.long, device=device)

        predicted_noise = model(x, t_batch)

        beta_t = betas[t_val].to(device)
        sqrt_recip_alpha_t = sqrt_recip_alpha[t_val].to(device)
        sqrt_one_minus_ab_t = sqrt_one_minus_alpha_bar[t_val].to(device)

        # Predict the mean of p(x_{t-1} | x_t)
        mean = sqrt_recip_alpha_t * (x - (beta_t / sqrt_one_minus_ab_t) * predicted_noise)

        if t_val > 0:
            noise = torch.randn_like(x)
            sigma_t = torch.sqrt(posterior_variance[t_val]).to(device)
            x = mean + sigma_t * noise
        else:
            x = mean

    model.train()
    return x.clamp(-1, 1)

In [ ]:
torch.manual_seed(42)
model.train()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_loss = 0.0
    for batch_idx, (x_0, _) in enumerate(train_loader):
        x_0 = x_0.to(device)
        t = torch.randint(0, T, (x_0.shape[0],), device=device)

        noise = torch.randn_like(x_0)
        x_t, _ = q_sample(x_0, t, noise=noise)

        predicted_noise = model(x_t, t)
        loss = F.mse_loss(predicted_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f}")

    if epoch in checkpoint_epochs:
        torch.manual_seed(0)  # fixed seed for comparable samples
        samples = ddpm_sample(model, n_samples=16)
        checkpoint_samples[epoch] = samples.cpu()
        model.train()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, NUM_EPOCHS + 1), losses, marker="o", linewidth=1.5)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training loss over epochs")
plt.tight_layout()
plt.show()

## 6. Generated Samples at Various Training Checkpoints

We saved samples at epochs 1, 5, 10, and 15. This shows how sample quality improves with training.

In [ ]:
fig, axes = plt.subplots(len(checkpoint_epochs), 8, figsize=(14, len(checkpoint_epochs) * 1.8))

for row, epoch in enumerate(checkpoint_epochs):
    samples = checkpoint_samples[epoch]
    for col in range(8):
        img = (samples[col].squeeze() + 1) / 2
        axes[row, col].imshow(img.numpy(), cmap="gray", vmin=0, vmax=1)
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(f"Epoch {epoch}", fontsize=10, rotation=0, labelpad=50)

plt.suptitle("Generated samples at training checkpoints", fontsize=13)
plt.tight_layout()
plt.show()

## 7. DDPM Sampling

Full DDPM sampling runs the reverse process for all $T=1000$ steps.

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\, \epsilon_\theta(x_t, t) \right) + \sigma_t\, z$$

where $z \sim \mathcal{N}(0, \mathbf{I})$ for $t > 0$ and $z = 0$ for $t = 0$.

In [ ]:
torch.manual_seed(42)
ddpm_samples = ddpm_sample(model, n_samples=16)

fig, axes = plt.subplots(2, 8, figsize=(14, 3.5))
for i in range(16):
    r, c = divmod(i, 8)
    img = (ddpm_samples[i].squeeze() + 1) / 2
    axes[r, c].imshow(img.cpu().numpy(), cmap="gray", vmin=0, vmax=1)
    axes[r, c].axis("off")

plt.suptitle("DDPM samples (T=1000 steps)", fontsize=13)
plt.tight_layout()
plt.show()

## 8. DDIM Sampling

DDIM (Song et al., 2020) is a deterministic sampling method that can use fewer steps.
The key update rule is:

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \left( \frac{x_t - \sqrt{1-\bar{\alpha}_t}\, \epsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}} \right) + \sqrt{1 - \bar{\alpha}_{t-1}}\, \epsilon_\theta(x_t, t)$$

No noise is added, making it deterministic and allowing us to skip timesteps.

In [ ]:
@torch.no_grad()
def ddim_sample(model, n_samples, ddim_steps=50, img_size=28, channels=1):
    """Generate samples using DDIM (deterministic, fewer steps).

    Args:
        model: trained noise prediction network
        n_samples: number of images to generate
        ddim_steps: number of denoising steps (can be much less than T)
    """
    model.eval()

    # Create a subsequence of timesteps
    step_size = T // ddim_steps
    timesteps = list(range(0, T, step_size))
    timesteps = list(reversed(timesteps))

    x = torch.randn(n_samples, channels, img_size, img_size).to(device)

    for i in range(len(timesteps)):
        t_val = timesteps[i]
        t_batch = torch.full((n_samples,), t_val, dtype=torch.long, device=device)

        predicted_noise = model(x, t_batch)

        ab_t = alpha_bar[t_val].to(device)
        sqrt_ab_t = torch.sqrt(ab_t)
        sqrt_one_minus_ab_t = torch.sqrt(1 - ab_t)

        # Predict x_0
        x_0_pred = (x - sqrt_one_minus_ab_t * predicted_noise) / sqrt_ab_t
        x_0_pred = x_0_pred.clamp(-1, 1)

        if i < len(timesteps) - 1:
            t_prev = timesteps[i + 1]
            ab_t_prev = alpha_bar[t_prev].to(device)
        else:
            ab_t_prev = torch.tensor(1.0).to(device)

        # DDIM deterministic step
        x = (torch.sqrt(ab_t_prev) * x_0_pred +
             torch.sqrt(1 - ab_t_prev) * predicted_noise)

    model.train()
    return x.clamp(-1, 1)

In [ ]:
torch.manual_seed(42)
ddim_50 = ddim_sample(model, n_samples=16, ddim_steps=50)

fig, axes = plt.subplots(2, 8, figsize=(14, 3.5))
for i in range(16):
    r, c = divmod(i, 8)
    img = (ddim_50[i].squeeze() + 1) / 2
    axes[r, c].imshow(img.cpu().numpy(), cmap="gray", vmin=0, vmax=1)
    axes[r, c].axis("off")

plt.suptitle("DDIM samples (50 steps)", fontsize=13)
plt.tight_layout()
plt.show()

### Compare DDIM at Different Step Counts

DDIM allows trading off quality for speed. Let's compare 10, 25, 50, and 100 steps.

In [ ]:
import time

ddim_step_counts = [10, 25, 50, 100]
ddim_results = {}

for steps in ddim_step_counts:
    torch.manual_seed(42)
    start = time.time()
    samples = ddim_sample(model, n_samples=8, ddim_steps=steps)
    elapsed = time.time() - start
    ddim_results[steps] = (samples.cpu(), elapsed)
    print(f"DDIM {steps:3d} steps: {elapsed:.2f}s")

In [ ]:
fig, axes = plt.subplots(len(ddim_step_counts), 8, figsize=(14, len(ddim_step_counts) * 1.8))

for row, steps in enumerate(ddim_step_counts):
    samples, elapsed = ddim_results[steps]
    for col in range(8):
        img = (samples[col].squeeze() + 1) / 2
        axes[row, col].imshow(img.numpy(), cmap="gray", vmin=0, vmax=1)
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(f"{steps} steps\n({elapsed:.1f}s)", fontsize=9, rotation=0, labelpad=55)

plt.suptitle("DDIM: quality vs number of denoising steps", fontsize=13)
plt.tight_layout()
plt.show()

### DDPM vs DDIM Side by Side

In [ ]:
torch.manual_seed(42)
start_ddpm = time.time()
ddpm_compare = ddpm_sample(model, n_samples=8)
ddpm_time = time.time() - start_ddpm

torch.manual_seed(42)
start_ddim = time.time()
ddim_compare = ddim_sample(model, n_samples=8, ddim_steps=50)
ddim_time = time.time() - start_ddim

fig, axes = plt.subplots(2, 8, figsize=(14, 3.5))
for col in range(8):
    img_ddpm = (ddpm_compare[col].squeeze().cpu() + 1) / 2
    axes[0, col].imshow(img_ddpm.numpy(), cmap="gray", vmin=0, vmax=1)
    axes[0, col].axis("off")

    img_ddim = (ddim_compare[col].squeeze().cpu() + 1) / 2
    axes[1, col].imshow(img_ddim.numpy(), cmap="gray", vmin=0, vmax=1)
    axes[1, col].axis("off")

axes[0, 0].set_ylabel(f"DDPM\n({ddpm_time:.1f}s)", fontsize=10, rotation=0, labelpad=45)
axes[1, 0].set_ylabel(f"DDIM 50\n({ddim_time:.1f}s)", fontsize=10, rotation=0, labelpad=45)

plt.suptitle("DDPM (1000 steps) vs DDIM (50 steps)", fontsize=13)
plt.tight_layout()
plt.show()

## 9. Visualize the Denoising Process Step by Step

We run DDPM sampling but record intermediate results to see how the model gradually turns noise into an image.

In [ ]:
@torch.no_grad()
def ddpm_sample_with_trajectory(model, n_samples=1, record_every=100, img_size=28, channels=1):
    """DDPM sampling that records intermediate states."""
    model.eval()
    x = torch.randn(n_samples, channels, img_size, img_size).to(device)
    trajectory = [(T, x.cpu().clone())]

    for t_val in reversed(range(T)):
        t_batch = torch.full((n_samples,), t_val, dtype=torch.long, device=device)

        predicted_noise = model(x, t_batch)

        beta_t = betas[t_val].to(device)
        sqrt_recip_alpha_t = sqrt_recip_alpha[t_val].to(device)
        sqrt_one_minus_ab_t = sqrt_one_minus_alpha_bar[t_val].to(device)

        mean = sqrt_recip_alpha_t * (x - (beta_t / sqrt_one_minus_ab_t) * predicted_noise)

        if t_val > 0:
            noise = torch.randn_like(x)
            sigma_t = torch.sqrt(posterior_variance[t_val]).to(device)
            x = mean + sigma_t * noise
        else:
            x = mean

        if t_val % record_every == 0:
            trajectory.append((t_val, x.cpu().clone()))

    model.train()
    return x.clamp(-1, 1), trajectory

In [ ]:
torch.manual_seed(42)
final_img, trajectory = ddpm_sample_with_trajectory(model, n_samples=4, record_every=100)

n_images = 4
n_steps = len(trajectory)

fig, axes = plt.subplots(n_images, n_steps, figsize=(n_steps * 1.5, n_images * 1.5))

for row in range(n_images):
    for col, (t_val, x_snap) in enumerate(trajectory):
        img = (x_snap[row].squeeze().clamp(-1, 1) + 1) / 2
        axes[row, col].imshow(img.numpy(), cmap="gray", vmin=0, vmax=1)
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(f"t={t_val}", fontsize=8)

plt.suptitle("Denoising trajectory: from pure noise to generated digits", fontsize=13)
plt.tight_layout()
plt.show()

### Detailed View: Last 100 Steps

Most of the structural detail emerges in the final denoising steps.

In [ ]:
torch.manual_seed(42)
_, trajectory_fine = ddpm_sample_with_trajectory(model, n_samples=1, record_every=10)

# Show the last ~100 steps (t=100 down to t=0)
fine_steps = [(t, x) for t, x in trajectory_fine if t <= 100]

fig, axes = plt.subplots(1, len(fine_steps), figsize=(len(fine_steps) * 1.2, 1.5))
for col, (t_val, x_snap) in enumerate(fine_steps):
    img = (x_snap[0].squeeze().clamp(-1, 1) + 1) / 2
    axes[col].imshow(img.numpy(), cmap="gray", vmin=0, vmax=1)
    axes[col].axis("off")
    axes[col].set_title(f"t={t_val}", fontsize=7)

plt.suptitle("Fine-grained denoising: t=100 to t=0", fontsize=12)
plt.tight_layout()
plt.show()

### What the Model Predicts: Noise vs Predicted x_0

At each timestep, the model predicts noise. We can use that to estimate $x_0$ and see what the model "thinks" the clean image looks like at various stages.

In [ ]:
@torch.no_grad()
def visualize_predictions(model, x_0, timesteps_to_show):
    """Show the model's noise prediction and x_0 estimate at various timesteps."""
    model.eval()
    n = len(timesteps_to_show)

    fig, axes = plt.subplots(3, n, figsize=(n * 1.8, 5))
    row_labels = ["Noisy x_t", "Predicted noise", "Estimated x_0"]

    torch.manual_seed(42)
    for col, t_val in enumerate(timesteps_to_show):
        t_tensor = torch.tensor([t_val]).to(device)
        x_0_dev = x_0.to(device)

        x_t, true_noise = q_sample(x_0_dev, t_tensor)
        pred_noise = model(x_t, t_tensor)

        # Estimate x_0 from x_t and predicted noise
        ab_t = alpha_bar[t_val].to(device)
        x_0_est = (x_t - torch.sqrt(1 - ab_t) * pred_noise) / torch.sqrt(ab_t)
        x_0_est = x_0_est.clamp(-1, 1)

        imgs = [x_t, pred_noise, x_0_est]
        for row in range(3):
            img = imgs[row][0].squeeze().cpu()
            if row == 1:  # noise: normalize for display
                img = (img - img.min()) / (img.max() - img.min() + 1e-8)
            else:
                img = (img + 1) / 2
            axes[row, col].imshow(img.numpy(), cmap="gray", vmin=0, vmax=1)
            axes[row, col].axis("off")
            if col == 0:
                axes[row, 0].set_ylabel(row_labels[row], fontsize=9, rotation=0, labelpad=75)
        axes[0, col].set_title(f"t={t_val}", fontsize=10)

    plt.suptitle("Model predictions at different timesteps", fontsize=13)
    plt.tight_layout()
    plt.show()
    model.train()


x_0_viz = sample_batch[0:1]
visualize_predictions(model, x_0_viz, [10, 50, 100, 250, 500, 750, 999])

## 10. Summary and Key Takeaways

What we implemented:

- **Forward process**: Gradually adds Gaussian noise to data according to a linear beta schedule. The closed-form expression using $\bar{\alpha}_t$ lets us jump to any timestep directly.

- **U-Net noise predictor**: A small encoder-decoder network with residual blocks and sinusoidal timestep embeddings. Takes noisy $x_t$ and timestep $t$, outputs predicted noise $\hat{\epsilon}$.

- **DDPM training**: Simple MSE loss between true noise and predicted noise. The model learns to denoise at all noise levels simultaneously.

- **DDPM sampling**: Iterates all $T$ reverse steps, adding stochasticity at each step. Produces diverse samples but is slow.

- **DDIM sampling**: Deterministic variant that can skip timesteps. Produces comparable quality with 20x fewer steps.

Key observations:
- The model's estimate of $x_0$ is blurry at high noise levels but sharpens as $t$ decreases.
- Most structural detail emerges in the final denoising steps.
- DDIM with 50 steps produces results comparable to DDPM with 1000 steps, at a fraction of the compute.